Вот конвертация HTML → Markdown:

---

Notebooks с семинара можно найти по [ссылке 1](https://colab.research.google.com/github/huggingface/cookbook/blob/main/notebooks/en/multimodal_rag_using_document_retrieval_and_vlms.ipynb), [ссылке 2](https://colab.research.google.com/drive/1J1BdJRB4EEpmWBrW5t9cDK-0yD-DEIBP?usp=sharing).

### Домашнее задание 12

**1. Часть 1 (5 баллов):**

* Заполнить дизайн-документ для RAG (см. [шаблон](https://colab.research.google.com/drive/1KBG74ca-fLKqORUZxfLiIQiNP1FvH0ZN?usp=sharing))

**2. Часть 2 (5 баллов):**

*На выбор два варианта:*

* Попробовать завести графовый RAG (FastRAG, Light-RAG, mini-RAG) на ранее собранных данных, замерить результаты ретрива и протестировать генерацию
* Попробовать завести мультимодальный RAG на любого рода картинках, инструкциях и протестировать на собранных запросах

> В обоих случаях объяснить выбор метода подсчёта метрик и предоставить эксперименты (опробовать различные библиотеки, модели, промпты).

**Домашнее задание необходимо предоставить в формате ссылки на Google Collab/Jupiter Notebook с вашими действиями и ключевыми выводами:**

* 24 ноября 23:59 — мягкий дедлайн
* 1 декабря 23:59 — жесткий дедлайн

До мягкого дедлайна за работу можно получить 10 баллов, после — 5; работы, отправленные после 1 декабря, могут быть проверены преподавателями до конца курса в формате зачет/не зачет.

---

Если нужно — могу автоматически конвертировать и другие HTML-фрагменты.


**1. Часть 1 (5 баллов):**

* Заполнить дизайн-документ для RAG (см. [шаблон](https://colab.research.google.com/drive/1KBG74ca-fLKqORUZxfLiIQiNP1FvH0ZN?usp=sharing))



# Design-Doc: Чат-бот / RAG-система

*(Заполняется проектной командой: бизнес-заказчик, ML/инженеры, продуктовая команда)*

## 1. Контекст проекта

### 1.1 Бизнес-задача

* Сократить нагрузку на операторов и ускорить обработку клиентских запросов за счёт автоматизации до 60–80% типовых вопросов.
* Обеспечить быстрый и консистентный доступ к внутренним знаниям (регламенты, инструкции, FAQ) в едином интерфейсе.
* Повысить удовлетворённость пользователей за счёт точных, актуальных и персонализированных ответов.

### 1.2 Целевая аудитория и пользователи

* Внутренние сотрудники (служба поддержки, продажи, HR, операционные команды).
* Клиенты/партнёры.
* Сценарии: текстовый чат в веб-виджете, интерфейс в корпоративном портале, интеграция с мессенджерами (Teams/Telegram), возможное расширение на голосовой интерфейс.
* Нагрузка: 5–10k сессий/сутки, пики в рабочие часы 10:00–16:00.

### 1.3 Ограничения и допущения

* Не покрываем транзакционные операции (например, изменения в профиле клиента), только информационные ответы.
* Не обрабатываем чувствительные персональные данные без явного разрешения.
* Предполагаем наличие доступа к актуальным корпоративным документам и API.
* Инфраструктура — облако/гибрид; требования к безопасности — соответствует внутренним политикам.

---

## 2. Архитектура решения

### 2.1 Общая схема

* Фронтенд (чат-интерфейс) → Backend/API Gateway → RAG-сервис → Retrieval (векторное хранилище) + Knowledge Store → LLM → Post-processing (фильтры, форматирование).

```mermaid
flowchart LR
    A[Frontend chat]
    B[Backend / API Gateway]
    C[RAG service]
    D[Retrieval vector DB]
    E[Knowledge Store]
    F[LLM]
    G[Post-processing]

    A --> B --> C --> F --> G
    C --> D
    C --> E
```

* Основные компоненты: чат UI, API, сервис обработки диалогов, RAG-движок, хранилище данных, мониторинг.

### 2.2 Хранилище знаний / документооборот

* Источники: корпоративный Wiki, внутренние PDF, регламенты, базы знаний, экспорт из Jira/Confluence, внешние API.
* Предобработка: парсинг PDF/HTML, очистка, сегментация по смысловым блокам, извлечение метаданных.
* Индексация: создание embedding, добавление меток (дата, тип документа, владелец), сохранение в векторный индекс.
* Обновление: автоматическая синхронизация 1 раз в сутки + ручное обновление по запросу контент-менеджера.

### 2.3 Retrieval + Generation

* Retrieval: Qdrant; cosine similarity; embedding размер 768–1024; k=3–5.
* Generation: LLM (локальная Llama-3.1) с шаблонными промптами.
* Контроль ответа: фильтрация нерелевантных результатов по threshold; fallback на статические FAQ; детектор галлюцинаций через self-check prompt.

### 2.4 Интеграции и интерфейсы

* Интеграции: CRM, базы пользователей (SSO/LDAP), система логирования, мониторинг (Prometheus, Grafana), корпоративные мессенджеры.
* Протоколы: REST для запросов, WebSockets для real-time диалогов, Webhooks для событий.

### 2.5 Инфраструктура и развертывание

* Docker-контейнеры, оркестрация через Kubernetes (dev/test/prod).
* Вычисления: CPU для retrieval, GPU — для LLM (если локальная); объём хранилища — 50–200 GB.
* Безопасность: OAuth2/SSO, TLS, контроль доступа по ролям, аудит действий.
* Масштабирование: горизонтальное масштабирование сервисов API и RAG.

---

## 3. Данные и качество знаний

### 3.1 Сбор и предобработка данных

* Форматы: PDF, DOCX, HTML, TXT, структурированные базы данных.
* Предобработка: извлечение текста, очистка, токенизация, выделение сущностей, сегментация на чанки 400–1200 символов.
* Добавление метаданных: тип документа, раздел, дата загрузки, версия документа.

### 3.2 Векторизация и индексирование

* Embedding-модель: text-embedding-3-large или аналогичная мультиязычная.
* Индекс: HNSW (cosine), 1–3 реплики, шардирование по типу данных.
* Стратегия обновлений: инкрементальная индексация + еженедельная пересборка для консистентности.

### 3.3 Метрики качества знаний

* Покрытие — доля документов, доступных системе (цель 90%).
* Актуальность — документы не старше 30 дней; SLA обновлений — 24 часа.
* Консистентность — отсутствие дубликатов; проверка конфликтующих инструкций.

---

## 4. Модель и генерация

### 4.1 Выбор LLM и промптинг

* Используем GPT-4.1/GPT-5 (облако) либо Llama-3.1-70B (локально) — оптимальное соотношение качества/стоимости.
* Промпт-шаблон: инструкции + retrieved context + ограничения формата + стиль.
* Ограничения: 8k–32k токенов контекста, среднее время генерации < 2 сек.

### 4.2 Контроль качества ответов

* Метрики: точность (manual eval), полнота, релевантность, полезность.
* Оценка ошибок: каталогизация галлюцинаций, детекция пустых ответов.
* Механизмы: reranking retriever’а, fallback «Извините, не нашёл…», ручное ревью спорных случаев.

### 4.3 Обучение/дообучение

* Fine-tuning при необходимости на корпоративных диалогах: фильтрация данных, размеченные пары.
* Версионность через MLflow; rollback — через хранение предыдущей модели.

---

## 5. UX / пользовательский опыт

### 5.1 Сценарии взаимодействия

* Приветствие → формирование запроса → уточняющие вопросы → ответ.
* Поиск по знаниям (FAQ, регламенты), многотуровый диалог.
* Исключения: отсутствие ответа, некорректные запросы, эскалация к оператору.

### 5.2 Диалоговая логика

* Поддержка контекста: хранение последних 5–10 сообщений.
* Multi-turn управление: слоты, уточнения, краткие подсказки.
* Тон: нейтральный, профессиональный; поддержка мультиязычности.

### 5.3 Метрики UX

* Время до первого ответа (< 1.5 сек), CSAT, NPS, % повторных обращений.
* Логирование всех диалогов; сбор пользовательских оценок.

---

## 6. Безопасность, соответствие и этика

* Обработка данных по внутренним политикам безопасности.
* Фильтры токсичности и небезопасного контента.
* Прозрачность: маркировка, что пользователь общается с ботом.
* Минимизация хранения персональных данных, шифрование в покое и в транзите.

---

## 7. План внедрения и эксплуатации

### 7.1 Этапы проекта

* Фаза 0 — исследование, сбор требований, аудит документов (2 недели).
* Фаза 1 — MVP: базовое Retrieval + LLM, чат-интерфейс (4–6 недель).
* Фаза 2 — расширение: улучшенный retriever, обновление индекса, интеграции (6–8 недель).
* Фаза 3 — оптимизация и масштабирование, A/B-тесты (4 недели).

### 7.2 Поддержка и эксплуатация

* Ответственные: команда DevOps/SRE + ML команда + контент-менеджер.
* Мониторинг: SLA 99%, latency, cost per request, ошибки.
* Регулярное обновление знаний (ежедневно), ежемесячная ревизия качества.

---

## 8. Риски и допущения

* Недостаточная полнота базы знаний → низкая точность ответов.
* Возможные галлюцинации модели → риск недостоверных ответов.
* Ограничения бюджета для облачных LLM → необходимость локальной модели.
* Непроверенные допущения: качество исходных документов, готовность команд к интеграциям.

---


## **9. Бюджет и ресурсы**

### **Человеческие ресурсы**

* **1 ML инженер** (частично, 0.5–1 FTE)
* **1 NLP инженер / MLE**
* **1 Backend инженер**
* **1 DevOps/SRE** (0.2–0.4 FTE благодаря использованию локальных HPC/VPS)
* **1 Product Owner**
* **1 UX/UI дизайнер** (на этапе разработки)
* **1 Контент-менеджер** (обновление и ревизия базы знаний)

### **Технологические ресурсы**

* GPU/CPU кластеры (локальные или аренда GPU-серверов)
* Локальные LLM (Llama 3)
* Локальные хранилища (S3-совместимые: Yandex Object Storage, VK Cloud, Selectel)
* Векторные БД: Qdrant

### **Примерные затраты**

#### **CAPEX (разовые затраты на запуск)**

* Разработка RAG-системы, интеграции, UI, подготовка базы знаний
* **Диапазон: $27k – $72k**

#### **OPEX (ежемесячные расходы)**

* Запросы к LLM (локально или через региональные облака)
* Поддержка DevOps/ML, администрирование, обновление данных
* Хостинг сервисов, векторное хранилище
* **Диапазон: $1,200 – $6,000 / месяц**

### **ROI оценки**

* Снижение затрат на операторов: **20–40%**
* Ускорение обработки запросов: **×3–5**
* Повышение удовлетворённости клиентов: **+15–25%**
* **Окупаемость: 2–5 месяцев** после запуска MVP

---

## 10. Приложения

* Словарь терминов и аббревиатур
* Ссылки на требования безопасности и UX-гайды
* Диаграммы архитектуры и логики диалога
* Чек-лист готовности к запуску



## **9. Бюджет и ресурсы**

### **Человеческие ресурсы**

* **1 ML инженер** (частично, 0.5–1 FTE)
* **1 NLP инженер / MLE**
* **1 Backend инженер**
* **1 DevOps/SRE** (0.2–0.4 FTE благодаря использованию локальных HPC/VPS)
* **1 Product Owner**
* **1 UX/UI дизайнер** (на этапе разработки)
* **1 Контент-менеджер** (обновление и ревизия базы знаний)

### **Технологические ресурсы**

* GPU/CPU кластеры (локальные или аренда GPU-серверов)
* Локальные LLM (Llama 3)
* Локальные хранилища (S3-совместимые: Yandex Object Storage, VK Cloud, Selectel)
* Векторные БД: Qdrant

### **Примерные затраты**

#### **CAPEX (разовые затраты на запуск)**

* Разработка RAG-системы, интеграции, UI, подготовка базы знаний
* **Диапазон: $27k – $72k**

#### **OPEX (ежемесячные расходы)**

* Запросы к LLM (локально или через региональные облака)
* Поддержка DevOps/ML, администрирование, обновление данных
* Хостинг сервисов, векторное хранилище
* **Диапазон: $1,200 – $6,000 / месяц**

### **ROI оценки**

* Снижение затрат на операторов: **20–40%**
* Ускорение обработки запросов: **×3–5**
* Повышение удовлетворённости клиентов: **+15–25%**
* **Окупаемость: 2–5 месяцев** после запуска MVP



**2. Часть 2 (5 баллов):**

*На выбор два варианта:*

* Попробовать завести графовый RAG (FastRAG, Light-RAG, mini-RAG) на ранее собранных данных, замерить результаты ретрива и протестировать генерацию
* Попробовать завести мультимодальный RAG на любого рода картинках, инструкциях и протестировать на собранных запросах

> В обоих случаях объяснить выбор метода подсчёта метрик и предоставить эксперименты (опробовать различные библиотеки, модели, промпты).

In [1]:
!pip uninstall -y langchain langchain-core langchain-text-splitters langchain-qdrant langchain-huggingface
!pip uninstall -y qdrant-client sentence-transformers transformers huggingface-hub
!pip uninstall qdrant-client

Found existing installation: langchain 1.0.0
Uninstalling langchain-1.0.0:
  Successfully uninstalled langchain-1.0.0
Found existing installation: langchain-core 1.1.0
Uninstalling langchain-core-1.1.0:
  Successfully uninstalled langchain-core-1.1.0
Found existing installation: langchain-text-splitters 1.0.0
Uninstalling langchain-text-splitters-1.0.0:
  Successfully uninstalled langchain-text-splitters-1.0.0
Found existing installation: langchain-qdrant 1.0.0
Uninstalling langchain-qdrant-1.0.0:
  Successfully uninstalled langchain-qdrant-1.0.0
Found existing installation: langchain-huggingface 1.0.1
Uninstalling langchain-huggingface-1.0.1:
  Successfully uninstalled langchain-huggingface-1.0.1
Found existing installation: qdrant-client 1.16.0
Uninstalling qdrant-client-1.16.0:
  Successfully uninstalled qdrant-client-1.16.0
Found existing installation: sentence-transformers 2.6.0
Uninstalling sentence-transformers-2.6.0:
  Successfully uninstalled sentence-transformers-2.6.0
Found 

In [2]:
!pip install \
  tqdm \
  pandas \
  langchain==1.0.0 \
  langchain-huggingface==1.0.1 \
  langchain-qdrant==1.0.0 \
  langchain-text-splitters==1.0.0 \
  qdrant-client==1.16.0 \
  huggingface_hub==0.34.0 \
  sentence-transformers==2.6.0 \
  transformers==4.40.0 \
  fastembed


  Using cached langchain-1.0.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached langchain_huggingface-1.0.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached langchain_qdrant-1.0.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached langchain_text_splitters-1.0.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached qdrant_client-1.16.0-py3-none-any.whl.metadata (11 kB)
  Using cached huggingface_hub-0.34.0-py3-none-any.whl.metadata (14 kB)
  Using cached sentence_transformers-2.6.0-py3-none-any.whl.metadata (11 kB)
  Using cached transformers-4.40.0-py3-none-any.whl.metadata (137 kB)
  Using cached langchain_core-1.1.0-py3-none-any.whl.metadata (3.6 kB)
Using cached langchain-1.0.0-py3-none-any.whl (106 kB)
Using cached langchain_huggingface-1.0.1-py3-none-any.whl (27 kB)
Using cached huggingface_hub-0.34.0-py3-none-any.whl (558 kB)
Using cached langchain_qdrant-1.0.0-py3-none-any.whl (24 kB)
Using cached qdrant_client-1.16.0-py3-none-any.whl (328 kB)
Using cached langchain_text_splitters-

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import Qdrant
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document

/Users/sergey/Projects/GigaSchool/llm-engineer/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Step 1 — Text cleaning (procedural)
# ----------------------------------
# Usage: set `csv_path` to your CSV file, run the cell. It will produce `data_cleaned.csv`.

import re
import pandas as pd

# emoji regex (covers common ranges)
_emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map symbols
    "\U0001F1E0-\U0001F1FF"  # flags
    "]+",
    flags=re.UNICODE
)

def clean_text_proc(text: str) -> str:
    """Clean a single text string: remove URLs, markdown links, hashtags (keep words), emojis, and extra whitespace."""
    if not isinstance(text, str):
        return ""
    # remove URLs
    text = re.sub(r'https?://\S+', '', text)
    # convert markdown links [label](url) -> label
    text = re.sub(r'\[(.*?)\]\(.*?\)', lambda m: m.group(1), text)
    # remove leading/trailing hashes while keeping the word
    text = re.sub(r'#(\w+)', lambda m: m.group(1), text)
    # strip emojis
    text = _emoji_pattern.sub('', text)
    # collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Procedural flow
csv_path = 'data/channel_data.csv' 
cleaned_path = 'data/channel_data_cleaned.csv'

print('Loading', csv_path)
df = pd.read_csv(csv_path, parse_dates=['date'])
print('Rows loaded:', len(df))

print('Cleaning text...')
df['text_clean'] = df['text'].apply(clean_text_proc)

print('Sample cleaned texts:')
print(df[['id', 'text_clean']].head().to_string())

print('Saving cleaned CSV to', cleaned_path)
df.to_csv(cleaned_path, index=False)
print('Done.')


Loading data/channel_data.csv
Rows loaded: 1144
Cleaning text...
Sample cleaned texts:
     id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                text_clean
0  1201                                                                                                                                                                                        

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)


doc_list = text_splitter.split_documents([
    Document(
        page_content=row['text'],
        metadata={
            "title": row['id']
            }
        )
        for i, row in df.iterrows()
    ])
doc_list

[Document(metadata={'title': 1201}, page_content='До Стачки осталось совсем немного времени. Подробнее об этой конференции я писал в'),
 Document(metadata={'title': 1201}, page_content='я писал в [посте](https://t.me/ohmyflutter/1195), и мы даже успели разыграть пару билетов.'),
 Document(metadata={'title': 1201}, page_content='У тех, кому в розыгрыше не повезло, но поучаствовать хочется, все еще есть время даже до “late'),
 Document(metadata={'title': 1201}, page_content='время даже до “late bird” цен.'),
 Document(metadata={'title': 1201}, page_content='📌 Программа и билеты доступны по ссылке.\nhttps://spb25.nastachku.ru/\n\n#event'),
 Document(metadata={'title': 1198}, page_content='FlutterCon Europe 25 начался 🚀🚀🚀'),
 Document(metadata={'title': 1198}, page_content='Кстати организаторы обещали в этот раз появление докладов у них на ютуб канале прямо в день'),
 Document(metadata={'title': 1198}, page_content='канале прямо в день выступления.'),
 Document(metadata={'title': 1197}, pa

In [6]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")


print(device)

mps


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

# symetric model
embed_model_name = "BAAI/bge-m3"
embed_model_kwargs = {'device': device}
embed_encode_kwargs = {'normalize_embeddings': True}
query_encode_kwargs = {}

# asymetric model
"""
embed_model_name = "intfloat/multilingual-e5-large"
embed_model_kwargs = {'device': device}
encode_kwargs = {"prompt": "passage: "} # prompts
query_encode_kwargs = {"prompt": "query: "} # prompts
"""

embeddings = HuggingFaceEmbeddings(
    model_name=embed_model_name,
    model_kwargs=embed_model_kwargs,
    encode_kwargs=embed_encode_kwargs,
)

/Users/sergey/Projects/GigaSchool/llm-engineer/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
import pkg_resources

print("qdrant-client:", pkg_resources.get_distribution("qdrant-client").version)
print("langchain-qdrant:", pkg_resources.get_distribution("langchain-qdrant").version)
print("langchain:", pkg_resources.get_distribution("langchain").version)


qdrant-client: 1.16.0
langchain-qdrant: 1.0.0
langchain: 1.0.0


In [9]:
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")
qdrant = QdrantVectorStore.from_documents(
    doc_list,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    location=":memory:",
    collection_name="my_documents_2",
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="custom_vector",
    sparse_vector_name="custom_sparse_vector",
)


qdrant_retriever = qdrant.as_retriever(verbose=True)

In [10]:
qdrant_retriever.invoke("Как стать Flutter разработчиком?")

[Document(metadata={'title': 87, '_id': '41c82f4624e14278a83a47146646eb11', '_collection_name': 'my_documents_2'}, page_content='📌Нужно ли изучать Flutter и где искать работу Flutter-разработчику'),
 Document(metadata={'title': 87, '_id': 'c3e978f91e6b4c4d93a8716ce379597b', '_collection_name': 'my_documents_2'}, page_content='Flutter или стать Flutter-разработчиком, чтобы больше зарабатывать.'),
 Document(metadata={'title': 1152, '_id': '4d33e4fb604543f89552be0eb9543d01', '_collection_name': 'my_documents_2'}, page_content='Компания Founders ищет разработчика на фреймворке Flutter'),
 Document(metadata={'title': 1055, '_id': 'ac4fcd2a2ff2475d86260db912b3737a', '_collection_name': 'my_documents_2'}, page_content='Также как и [drawRawAtlas](https://api.flutter.dev/flutter/dart-ui/Canvas/drawRawAtlas.html)')]

## GRAPH RAG


## 🔎 Что такое LightRAG

* LightRAG — фреймворк для RAG, который использует **граф + векторы** для улучшенного retrieval. ([GitHub][1])
* Поддерживает разные back‑end для векторного хранилища, в том числе **QdrantVectorDBStorage**. ([GitHub][1])
* Предоставляет режимы запроса: `local`, `global`, `hybrid`, `mix` и др. через `QueryParam`. ([GitHub][1])
* Нужно вызвать `initialize_storages()` перед вставкой документов, чтобы инициализировать все сторы (вектор, граф, статус документов). ([GitHub][1])



## ✅ Установка LightRAG

In [11]:
#!pip uninstall lightrag lightrag-hku -y
!pip install lightrag-hku==1.4.9.8


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## 🧪 Пример кода: базовый pipeline LightRAG

Предположим, у тебя есть:

* `chunks_list` — список текстовых чанков или документов (строк)
* `embedding_func` — асинхронная функция, которая берёт список текстов и возвращает массив эмбеддингов
* `llm_model_func` — асинхронная функция, которая генерирует ответ LLM по промпту

Вот простой "ноутбук‑пример" (или фрагмент кода):


In [12]:
import asyncio

def run_async(coro):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        return asyncio.ensure_future(coro)
    else:
        return asyncio.run(coro)






```python

```

---

## ⚙️ Настройки `QueryParam`

При создании `QueryParam` ты можешь конфигурировать:

* `mode`: `"local" | "global" | "hybrid" | "mix" | "naive"` — определяет способ retrieval ([GitHub][1])
* `only_need_context`: если `True`, вернуть только контекст без генерации ответа. ([GitHub][1])
* `user_prompt`: можно передать дополнительный текст, который LLM использует после retrieval. ([GitHub][1])
* `enable_rerank`: включает reranking, если reranker настроен. ([GitHub][1])

---

## 🔧 Важные рекомендации

1. **Выбор модели эмбеддингов**
   Убедись, что embedding-функция возвращает эмбеддинги того же размера, который LightRAG ожидает (`embedding_dim`).

2. **Совместимость векторного хранилища**
   При `vector_storage="QdrantVectorDBStorage"` нужно, чтобы Qdrant был поднят, или использовать файл SQLite‑подобное решение, если поддерживается.

3. **Графовая часть**
   LightRAG строит граф сущностей / отношений — это добавляет “глубину” retrival. Поэтому, когда вставляешь текст: элементы (сущности) извлекаются, и граф обновляется.

4. **Переиндексация**
   Если меняешь embedding‑модель, Vector DB необходимо переинициализировать (удалить старые векторы) и заново проиндексировать, иначе размер эмбеддингов не будет соответствовать.


## 1. Jupyter Notebook — Полный эксперимент

### 1.1. Инициализация моделей

In [13]:
from sentence_transformers import SentenceTransformer

# Модель эмбеддингов ["sentence-transformers/all-mpnet-base-v2"]
embed_model = SentenceTransformer(embed_model_name)

EMBED_DIM = 1024  # размерность модели

async def embedding_func(texts):
    return embed_model.encode(texts, convert_to_numpy=True)

/Users/sergey/Projects/GigaSchool/llm-engineer/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [14]:
#import logging
#from transformers import AutoModelForCausalLM, AutoTokenizer
#import torch
#
## LLM
#LLM_NAME = "Qwen/Qwen3-4B-Instruct-2507"
#tokenizer = AutoTokenizer.from_pretrained(LLM_NAME, trust_remote_code=True)
#model = AutoModelForCausalLM.from_pretrained(
#    LLM_NAME,
#    #torch_dtype=torch.float16,
#    trust_remote_code=True,
#).to(device)
#
#async def llm_model_func(prompt: str, **kwargs) -> str:
#
#    # Логирование запроса
#    #print("DEBUG: LLM prompt for extraction:\n%s", extraction_prompt)
#
#    # Токенизация и генерация
#    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
#    output = model.generate(
#        **inputs,
#        max_new_tokens=512,
#        do_sample=True,
#        temperature=0.7,
#        top_k=50,
#        top_p=0.9
#    )
#    text = tokenizer.decode(output[0], skip_special_tokens=True)
#
#    # Логирование ответа от LLM
#    print("DEBUG: LLM raw output:\n%s", text)
#
#    # Проверка окончания — если нет `<|COMPLETE|>`, можно добавить вручную
#    if "<|COMPLETE|>" not in text:
#        #print("WARNIGN: LLM output does not contain completion delimiter `<|COMPLETE|>`; appending it.")
#        text = text.strip() + "\n<|COMPLETE|>"
#
#    #print("DEBUG: RESULT:\n%s", text)
#    return text


In [15]:
#!pip install -U transformers accelerate safetensors

In [16]:
import logging
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

LLM_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(
    LLM_NAME,
    revision="main",      # ← критично
    trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    revision="main",
    trust_remote_code=True,
    torch_dtype="auto"
).to(device)


ULTRA_STRICT_PROMPT = """You are an Information Extraction Engine operating in STRICT MODE.
You MUST output ONLY valid LightRAG extraction lines.

CRITICAL RULES (MUST FOLLOW):
1. Output ONLY the allowed formats below.
2. NO markdown, NO punctuation outside fields, NO bullets, NO commentary.
3. NO introductory text, NO explanations, NO blank lines.
4. DO NOT output anything before the first entity or relation line.
5. DO NOT generate any lines that do not strictly match the required formats.

VALID OUTPUT FORMATS (ONLY these):

For ENTITIES (exactly 4 fields):
entity#|#|<entity_name>#|#|<entity_type>#|#|<extra_info>

For RELATIONS (exactly 5 fields):
relation#|#|<relation_type>#|#|<subject_entity>#|#|<object_entity>#|#|<extra_info>

FIELD RULES:
- You MUST NOT use "#", "|", "#|" inside fields.
- Do NOT leave any field empty.
- <entity_type> MUST be one of: person, organization, location, concept, object, event, role.
- <relation_type> MUST be a simple verb-like string (e.g., "uses", "creates", "belongs_to", "mentions").

TERMINATION:
After ALL extraction lines, output EXACTLY:
<|COMPLETE|>

If NOTHING is extractable, output ONLY:
<|COMPLETE|>

Begin extraction now.
"""


# -------------------------------------------------------------
# STRICT POSTPROCESSOR — гарантирует корректность LightRAG формата
# -------------------------------------------------------------
def fix_extraction_output(text: str) -> str:

    # убираем старые COMPLETE
    text = text.replace("<|COMPLETE|>", "")

    lines = text.splitlines()
    out = []

    # исправленные регулярки
    entity_pattern = re.compile(
        r"^entity#\|\#\|(.+?)#\|\#\|(.+?)#\|\#\|(.*)$"
    )
    relation_pattern = re.compile(
        r"^relation#\|\#\|(.+?)#\|\#\|(.+?)#\|\#\|(.+?)#\|\#\|(.*)$"
    )

    for line in lines:
        l = line.strip()
        if entity_pattern.match(l) or relation_pattern.match(l):
            out.append(l)

    # окончательный terminator
    out.append("<|COMPLETE|>")
    return "\n".join(out)


# -------------------------------------------------------------
# UNIVERSAL LLM FUNCTION
# -------------------------------------------------------------
async def llm_model_func(prompt: str, task: str = None, **kwargs) -> str:

    # prepend strict system prompt for extraction
    if task == "extract":
        prompt = ULTRA_STRICT_PROMPT + "\n" + prompt

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.1 if task == "extract" else 0.5,
        top_k=40,
        top_p=0.9
    )

    raw_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # strict extraction mode
    if task == "extract":
        return fix_extraction_output(raw_text)

    # normal mode
    return raw_text + "\n<|COMPLETE|>"


Exception: data did not match any variant of untagged enum ModelWrapper at line 757479 column 3

In [ ]:
text_chunk = """
Недавно в языке появилась такая вещь как Native Assets, но что это такое и зачем это нужно?

На эти и некоторые другие вопросы отвечает автор сегодняшней статьи. В ней достаточно подробно описаны ключевые вещи, чтобы появилось базовое понимание зачем нужны Native Assets, что такое Build Hooks и как это всё между собой подружить.

Также неплохим начальным шагом будет описание от самой команды Dart.
"""

prompt = f"""
Extract all entities and relationships from the text below.
Text: {soft_clean(text_chunk)}

Output format MUST be:
entity#|#|<entity_name>#|#|<entity_type>#|#|<extra_info>
relation#|#|<relation_type>#|#|<subject_entity>#|#|<object_entity>#|#|<extra_info>

If nothing found, output only <|COMPLETE|>.
"""
entities = await llm_model_func(prompt, task="extract")
entities

### 1.2. Загружаем CSV

In [ ]:
import pandas as pd

#csv_path = "your_dataset.csv"
#df = pd.read_csv(csv_path, parse_dates=["date"])
df.head()


In [ ]:
df.shape

## 2. Semantic chunking + загрузка в LightRAG

In [ ]:
!pip install semantic-text-splitter


In [ ]:
!pip install clean-text

In [ ]:
!pip install faiss-cpu

In [ ]:

from cleantext import clean

def soft_clean(text: str) -> str:
    return clean(
        text,
        fix_unicode=True,               # исправляем юникод
        to_ascii=False,                 # оставляем кириллицу
        lower=False,                    # сохраняем регистр
        no_line_breaks=False,           # сохраняем переносы
        no_urls=True,                   # убираем ссылки, если нужно
        no_emails=True,
        no_phone_numbers=True,
        no_numbers=False,
        no_digits=False,
        no_currency_symbols=True,
        no_punct=False                  # сохраняем знаки препинания!
    )

def smart_split(text: str, splitter):
    if len(text) < 200:
        return [text]  # короткий текст без деления
    return splitter.split_text(text)

In [ ]:
import requests
import time
import json

async def chat_completion(
    messages,
    model="gpt-4.1",
    max_tokens=200,
    temperature=0.7,
    url="http://localhost:8000/v1/chat/completions",
    delay=0.5,
    timeout=30,
    **kwargs
):
    """
    Универсальный обёртка для OpenAI совместимых моделей (FastAPI / LM Studio / llama.cpp server).
    """
    time.sleep(delay)

    headers = {"Content-Type": "application/json"}
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }

    resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
    resp.raise_for_status()
    return resp.json()


# =====================================================
# === MAIN LLM for LightRAG ===========================
# =====================================================

import re
import asyncio
from typing import Optional

STRICT_EXTRACTION_SYSTEM_PROMPT = """
You MUST output ONLY valid LightRAG extraction lines.
Forbidden: markdown, prose, bullets, explanations, blank lines.

Allowed formats ONLY:

entity#|#|<entity_name>#|#|<entity_type>#|#|<extra_info>
relation#|#|<relation_type>#|#|<subject_entity>#|#|<object_entity>#|#|<extra_info>

Rules:
1. Every line must start with "entity#|#|" or "relation#|#|".
2. There must be EXACTLY 4 fields for entity, separated by "#|#|".
3. There must be EXACTLY 5 fields for relation, separated by "#|#|".
4. No other characters are allowed before, after, or between fields.
5. After all lines, output exactly "<|COMPLETE|>".
6. If no entities/relations found — output only "<|COMPLETE|>".

If you break ANY rule — the system fails. Do not break rules.
"""

def fix_extraction_output(text: str) -> str:
    out = []

    # entity: entity#|#|A#|#|B#|#|C
    entity_pattern = re.compile(
        r"^entity#\|\#\|([^#]+)#\|\#\|([^#]+)#\|\#\|([^#]+)#\|\#\|([^#]+)$"
    )

    # relation: relation#|#|A#|#|B#|#|C#|#|D#|#|E
    relation_pattern = re.compile(
        r"^relation#\|\#\|([^#]+)#\|\#\|([^#]+)#\|\#\|([^#]+)#\|\#\|([^#]+)#\|\#\|([^#]+)$"
    )

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue

        if entity_pattern.match(line) or relation_pattern.match(line):
            out.append(line)

    out.append("<|COMPLETE|>")
    return "\n".join(out)



async def my_llm(prompt: str, task: str = "default", **kwargs) -> str:
    """
    Универсальная LLM-функция для LightRAG
    task: str - режим работы: "extract", "summarize", "qa" и т.д.
    """
    
    if task == "extract":
        system_prompt = STRICT_EXTRACTION_SYSTEM_PROMPT
    elif task == "summarize":
        system_prompt = "You are a summarizer. Summarize the user's input concisely."
    elif task == "qa":
        system_prompt = "You are a QA assistant. Answer the user's question based on context."
    else:
        system_prompt = "You are a helpful assistant."

    # Формируем messages для LLM
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    # Вызываем LLM через HTTP (или OpenAI-compatible API)
    resp = await chat_completion(messages=messages, **kwargs)

    if resp is None:
        return "<|COMPLETE|>" if task == "extract" else ""

    try:
        raw = resp["choices"][0]["message"]["content"]
    except Exception:
        return "<|COMPLETE|>" if task == "extract" else ""

    # Постобработка только для extract
    if task == "extract":
        return fix_extraction_output(raw)
    else:
        return raw



In [ ]:
text_chunk = """
Недавно в языке появилась такая вещь как Native Assets, но что это такое и зачем это нужно?

На эти и некоторые другие вопросы отвечает автор сегодняшней статьи. В ней достаточно подробно описаны ключевые вещи, чтобы появилось базовое понимание зачем нужны Native Assets, что такое Build Hooks и как это всё между собой подружить.

Также неплохим начальным шагом будет описание от самой команды Dart.
"""

prompt = f"""
Extract all entities and relationships from the text below.
Text: {soft_clean(text_chunk)}

Output format MUST be:
entity#|#|<entity_name>#|#|<entity_type>#|#|<extra_info>
relation#|#|<relation_type>#|#|<subject_entity>#|#|<object_entity>#|#|<extra_info>

If nothing found, output only <|COMPLETE|>.
"""
entities = await my_llm(prompt, task="extract")
entities


In [ ]:
custom_extract_prompt = """
SYSTEM: 
You MUST output ONLY valid LightRAG extraction lines.
Forbidden: markdown, prose, bullets, explanations, blank lines.

Allowed formats ONLY:

entity#|#|<entity_name>#|#|<entity_type>#|#|<extra_info>
relation#|#|<relation_type>#|#|<subject_entity>#|#|<object_entity>#|#|<extra_info>

Rules:
1. Every line must start with "entity#|#|" or "relation#|#|".
2. There must be EXACTLY 4 fields for entity, separated by "#|#|".
3. There must be EXACTLY 5 fields for relation, separated by "#|#|".
4. No other characters are allowed before, after, or between fields.
5. After all lines, output exactly "<|COMPLETE|>".
6. If no entities/relations found — output only "<|COMPLETE|>".

If you break ANY rule — the system fails. Do not break rules.
"""

#addon_params = {
#    "override_extract_prompt": custom_extract_prompt
#}

In [ ]:
from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc
from lightrag.kg.shared_storage import initialize_pipeline_status
#from semantic_text_splitter import TextSplitter
from cleantext import clean
from tqdm.auto import tqdm
import asyncio
import shutil

# Semantic splitter
#splitter = TextSplitter(chunk_size=400)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Убить каталог storage LightRAG
shutil.rmtree("./data/lightrag_storage", ignore_errors=True)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=50,
)

addon_params = {
    #"custom_text_splitter": splitter,
    "override_extract_prompt": custom_extract_prompt  # если ты уже используешь prompt
}


# LightRAG instance
rag = LightRAG(
    working_dir="./data/lightrag_storage",
    addon_params=addon_params,
    llm_model_func=llm_model_func,
    #llm_model_func=my_llm,
    embedding_func=EmbeddingFunc(embedding_dim=EMBED_DIM, func=embedding_func),
    #vector_storage="QdrantVectorDBStorage",  # используем Qdrant
    vector_storage="FaissVectorDBStorage",
    #splitter=splitter,
)

#await rag.aclear_cache()

await rag.initialize_storages()
await initialize_pipeline_status()


In [ ]:
emb = await embedding_func("тест текст")
print(len(emb))  # или emb.shape, если это numpy


### 2.1. Функция загрузки датасета

In [ ]:

    
async def load_dataset_into_lightrag(rag, df):
    all_chunks = []

    for _, row in df.iterrows():
        text = row["text_clean"]

        # Приводим текст к строке
        if not isinstance(text, str):
            text = str(text)
            
        #print(f"DEBUG: RAW TEXT: {text}")

        # Clean
        text = soft_clean(text)
        
        #print(f"DEBUG: CLEAR TEXT: {text}")
        

        # Split
        chunks = smart_split(text, splitter)
        #chunks = splitter.split_text(text)

        # Гарантия: все чанки строки
        chunks = [str(c) for c in chunks]

        all_chunks.extend(chunks)

        # Batch insert
        if len(all_chunks) >= 100:
            await rag.ainsert(list(all_chunks))
            all_chunks = []

    # final batch
    if all_chunks:
        await rag.ainsert(list(all_chunks))


### 2.2. Запуск импорта

In [ ]:
await load_dataset_into_lightrag(rag, df[:5])


In [ ]:
kg = await rag.get_knowledge_graph(node_label="Entity", max_depth=2, max_nodes=2)
print(kg.nodes)
print(kg.edges)




In [ ]:
from lightrag import QueryParam

result = await rag.aquery("Flutter", QueryParam(mode="global"))
if hasattr(result, "get_response_text"):
    text = result.get_response_text()
    print("Ответ:", text)
elif hasattr(result, "retriever_output"):
    print("Документы:", result.retriever_output.docs)
else:
    print("Результат:", result)




In [ ]:
result = await rag.aquery("Flutter", QueryParam(mode="global"))

In [ ]:
import logging
from lightrag import QueryParam

# Настройка логирования
logger = logging.getLogger("my_lightrag")
logger.setLevel(logging.DEBUG)
handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s — %(name)s — %(levelname)s — %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

async def query_rag(rag, query_str: str):
    logger.debug(f"Запрос: {query_str}")
    try:
        # Выполняем запрос
        param = QueryParam(mode="global", top_k=5, chunk_top_k=3)
        result = await rag.aquery(query_str, param)
        logger.debug(f"Raw result type: {type(result)}, content: {result}")

        # Проверяем, что это список
        if isinstance(result, list):
            # Логируем каждый элемент
            for i, item in enumerate(result):
                logger.debug(f"Result[{i}]: {item}")
            # Можно сформировать одну строку из всех элементов
            combined = "\n".join(str(item) for item in result)
            return combined
        else:
            # Если возвращается не список — просто вернуть как строку
            return str(result)
    except Exception as e:
        logger.error("Ошибка при запросе RAG:", exc_info=e)
        return None

# Пример вызова
res = await query_rag(rag, "Flutter")
print("Ответ RAG:", res)


In [ ]:
llm_model_func("Flutter")

In [ ]:
df["text"][5]

In [ ]:
import langchain
print(langchain.__version__)


# vFull

In [ ]:
import asyncio
import re
import pandas as pd
import torch
from lightrag import LightRAG
#from langchain.embeddings.huggingface import HuggingFaceEmbeddings
#from langchain.embeddings import HuggingFaceEmbeddings


# -----------------------------
# Устройство для модели
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# Qwen LLM
# -----------------------------
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_NAME = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    trust_remote_code=True,
).to(device)

# -----------------------------
# ULTRA-STRICT Extraction Prompt
# -----------------------------
ULTRA_STRICT_PROMPT = """ 
You are an Information Extraction Engine operating in STRICT MODE.
You MUST output ONLY valid LightRAG extraction lines.

CRITICAL RULES (MUST FOLLOW):
1. Output ONLY the allowed formats below.
2. NO markdown, NO punctuation outside fields, NO bullets, NO commentary.
3. NO introductory text, NO explanations, NO blank lines.
4. DO NOT output anything before the first entity or relation line.
5. DO NOT generate any lines that do not strictly match the required formats.

VALID OUTPUT FORMATS (ONLY these):

For ENTITIES (exactly 4 fields):
entity#|#|<entity_name>#|#|<entity_type>#|#|<extra_info>

For RELATIONS (exactly 5 fields):
relation#|#|<relation_type>#|#|<subject_entity>#|#|<object_entity>#|#|<extra_info>

FIELD RULES:
- You MUST NOT use "#", "|", "#|" inside fields.
- Do NOT leave any field empty.
- <entity_type> MUST be one of: person, organization, location, concept, object, event, role.
- <relation_type> MUST be a simple verb-like string (e.g., "uses", "creates", "belongs_to", "mentions").

TERMINATION:
After ALL extraction lines, output EXACTLY:
<|COMPLETE|>

If NOTHING is extractable, output ONLY:
<|COMPLETE|>

IF YOU VIOLATE ANY RULE → the system becomes invalid. Never violate the rules.

Begin extraction now.
"""

# -----------------------------
# STRICT POSTPROCESSOR
# -----------------------------
def fix_extraction_output(text: str) -> str:
    lines = text.splitlines()
    out = []

    entity_pattern = re.compile(r"^entity#\|\#\|[^#]+#\|\#\|[^#]+#\|\#\|.*$")
    relation_pattern = re.compile(r"^relation#\|\#\|[^#]+#\|\#\|[^#]+#\|\#\|[^#]+#\|\#\|.*$")

    for line in lines:
        line = line.strip()
        if entity_pattern.match(line) or relation_pattern.match(line):
            out.append(line)

    out.append("<|COMPLETE|>")
    return "\n".join(out)

# -----------------------------
# UNIVERSAL LLM FUNCTION
# -----------------------------
async def llm_model_func(prompt: str, task: str = None, **kwargs) -> str:
    if task == "extract":
        prompt = ULTRA_STRICT_PROMPT + "\n" + prompt

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.1 if task == "extract" else 0.5,
        top_k=40,
        top_p=0.9
    )

    raw_text = tokenizer.decode(output[0], skip_special_tokens=True)

    if task == "extract":
        return fix_extraction_output(raw_text)
    else:
        return raw_text + "\n<|COMPLETE|>"

# -----------------------------
# SPLITTER
# -----------------------------
#from langchain.text_splitter import RecursiveCharacterTextSplitter
#
#splitter = RecursiveCharacterTextSplitter(
#    chunk_size=100,
#    chunk_overlap=30
#)
def simple_split(text, chunk_size=200, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


# -----------------------------
# Embeddings
# -----------------------------
from sentence_transformers import SentenceTransformer

# Инициализация модели эмбеддингов
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")  # или "cuda"
def embedding_func(text: str):
    return embedding_model.encode(text, convert_to_tensor=True).cpu().numpy()



class EmbeddingWrapper:
    def __init__(self, func, dim):
        self.func = func
        self.embedding_dim = dim

    async def __call__(self, text: str):
        # Если твой func sync, можно обернуть в asyncio
        if callable(getattr(self.func, "__call__", None)):
            return self.func(text)
        else:
            return self.func(text)

# допустим, твоя функция embedding_func(text) возвращает вектор длины 384
embedding_func_wrapped = EmbeddingWrapper(embedding_func, dim=384)

# -----------------------------
# INIT LightRAG
# -----------------------------
def init_lightrag(work_dir: str = "rag_data") -> LightRAG:
    rag = LightRAG(
        working_dir=work_dir,
        llm_model_func=llm_model_func,
        embedding_func=embedding_func_wrapped,
        vector_storage="FaissVectorDBStorage"
    )
    return rag

# -----------------------------
# SMART SPLIT FUNCTION
# -----------------------------
def smart_split(text, splitter):
    return splitter.split_text(text)

# -----------------------------
# SOFT CLEAN
# -----------------------------
def soft_clean(text: str) -> str:
    return text.strip().replace("\n", " ").replace("\r", " ")

# -----------------------------
# LOAD DATA INTO LightRAG
# -----------------------------
async def load_dataset_into_lightrag(rag, df):
    all_chunks = []

    for _, row in df.iterrows():
        text = row["text_clean"]
        if not isinstance(text, str):
            text = str(text)

        text = soft_clean(text)
        #chunks = smart_split(text, splitter)
        chunks = simple_split(text, chunk_size=200, overlap=50)
        chunks = [str(c) for c in chunks]

        all_chunks.extend(chunks)

        # Batch insert
        if len(all_chunks) >= 100:
            await rag.ainsert(list(all_chunks))
            all_chunks = []

    if all_chunks:
        await rag.ainsert(list(all_chunks))

# -----------------------------
# MAIN
# -----------------------------
async def main():
    df = pd.DataFrame({"text_clean": [
        "Apple Inc. is a company based in Cupertino. Tim Cook is the CEO.",
        "Gucci is a fashion brand. Its logo features a stylized G."
    ]})

    rag = init_lightrag()
    await rag.initialize_storages()
    await load_dataset_into_lightrag(rag, df[:10])

    # Пример запроса
    query = "Какие организации и персоны упомянуты?"
    results = await rag.aquery(query)
    print("=== RAG QUERY RESULT ===")
    print(results)

# -----------------------------
# EXECUTE
# -----------------------------
# В Jupyter используем:
# await main()

# В обычном скрипте:
# if __name__ == "__main__":
#     asyncio.run(main())


In [ ]:
await main()


In [ ]:
import shutil
import asyncio
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc
from lightrag.kg.shared_storage import initialize_pipeline_status

from langchain_text_splitters import RecursiveCharacterTextSplitter

# -------------------------------
# 1. Эмбеддинги
# -------------------------------
embed_model_name = "sentence-transformers/all-mpnet-base-v2"
embed_model = SentenceTransformer(embed_model_name)
EMBED_DIM = 1024

async def embedding_func(texts):
    return embed_model.encode(texts, convert_to_numpy=True)


# -------------------------------
# 2. LLM
# -------------------------------
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    trust_remote_code=True,
).to(device)

async def llm_model_func(prompt: str, **kwargs) -> str:
    """Генерация ответа LLM с delimiter для LightRAG"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    output = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)

    # Добавляем delimiter если его нет
    if "<|COMPLETE|>" not in text:
        text = text.strip() + "\n<|COMPLETE|>"

    return text


# -------------------------------
# 3. Очистка storage
# -------------------------------
shutil.rmtree("./data/lightrag_storage", ignore_errors=True)


# -------------------------------
# 4. Сплиттер для коротких текстов
# -------------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)


# -------------------------------
# 5. LightRAG instance
# -------------------------------
rag = LightRAG(
    working_dir="./data/lightrag_storage",
    addon_params=addon_params,  # твои параметры
    llm_model_func=llm_model_func,
    embedding_func=EmbeddingFunc(embedding_dim=EMBED_DIM, func=embedding_func),
    vector_storage="FaissVectorDBStorage",
    text_splitter=splitter  # обязательно передаем сплиттер
)


# -------------------------------
# 6. Инициализация
# -------------------------------
await rag.initialize_storages()
await initialize_pipeline_status()


# -------------------------------
# 7. Пример запроса
# -------------------------------
text = "Alice joined Acme Corp in 2023 as a software engineer."
query = QueryParam(
    query=text,
    top_k=3,  # количество релевантных документов
)
response = await rag.query(query)
print(response)


In [ ]:
# ---------------------------
# 1) Кастомные промпты
# ---------------------------

addon_params = {
    "entity_types": [
        "event",
        "person",
        "organization",
        "technology",
        "product",
        "media",
        "topic",
        "tool",
        "framework",
        "other"
    ],

    "entity_extraction": """
---Role---
You are a Knowledge Graph Specialist extracting entities and relationships specifically from IT-media posts, conference updates, video announcements, technical episodes, and news about technologies.

Your goal: extract clean structured entities and relationships from texts related to Flutter, Dart, conferences, tools, frameworks, AI-features, guests, episodes, releases, showcases and developer tooling.

---Instructions---

1. Entity Extraction
   Extract only **meaningful, real entities**:
   - Events (FlutterCon Europe 25)
   - People (speakers, guests)
   - Organizations (Google, channels)
   - Technologies (Flutter, Figma MCP)
   - Media (videos, episodes)
   - Topics (theming, typography)
   - Tools (plugins, analyzers)
   - Frameworks (Flutter, AngularDart)

Format:
entity{tuple_delimiter}entity_name{tuple_delimiter}entity_type{tuple_delimiter}entity_description

2. Relationship Extraction
Format:
relation{tuple_delimiter}source{tuple_delimiter}target{tuple_delimiter}keywords{tuple_delimiter}description

Allowed relations:
- participates_in
- hosted_by
- guest_in
- demonstrates
- uses
- explains
- covers
- related_to
- publishes
- mentions
- part_of

3. Rules:
- Output only in {language}
- Use EXACT entity names
- Output entities first, then relationships
- End with {completion_delimiter}

---Examples---

entity{tuple_delimiter}FlutterCon Europe 25{tuple_delimiter}event{tuple_delimiter}Major European Flutter conference in 2025.
entity{tuple_delimiter}Observable Flutter{tuple_delimiter}media{tuple_delimiter}Flutter weekly show.
entity{tuple_delimiter}Muhammad Hamza{tuple_delimiter}person{tuple_delimiter}Guest of the episode.
entity{tuple_delimiter}Figma MCP{tuple_delimiter}technology{tuple_delimiter}AI-powered design-system generator.

relation{tuple_delimiter}Observable Flutter{tuple_delimiter}Muhammad Hamza{tuple_delimiter}guest_in{tuple_delimiter}Guest of the episode.
relation{tuple_delimiter}Muhammad Hamza{tuple_delimiter}Figma MCP{tuple_delimiter}demonstrates{tuple_delimiter}Shows how Figma MCP works.

{completion_delimiter}

---Real Data---
Entity_types: [{entity_types}]
Text:
```

{input_text}

```
""",

    "entity_continue_extraction": """
---Role---
You continue extraction only for missing or incorrectly formatted items.

---Instructions---

1. DO NOT re-output previously correct items.
2. Output only:
   - Missing entities
   - Missing relationships
   - Incorrect/truncated items

3. Format MUST be identical:
   entity{tuple_delimiter}...(4 fields)
   relation{tuple_delimiter}...(5 fields)

4. End with {completion_delimiter}.
5. Output only data, no comments.

---Real Data---
Entity_types: [{entity_types}]
Text:
```

{input_text}

```
"""
}

Вот **полный рабочий Python-скрипт**, который:

✔ создаёт `addon_params`
✔ запускает LightRAG
✔ добавляет документы
✔ запускает async-pipeline
✔ выполняет запросы

Полностью готов к копированию и запуску.

---

# ✅ **Полный Python-скрипт с кастомным промптом LightRAG**

```python
import asyncio
import logging
from lightrag import LightRAG, QueryParam

logging.basicConfig(level=logging.DEBUG)

# ---------------------------
# 1) Кастомные промпты
# ---------------------------

addon_params = {
    "entity_types": [
        "event",
        "person",
        "organization",
        "technology",
        "product",
        "media",
        "topic",
        "tool",
        "framework",
        "other"
    ],

    "entity_extraction": """
---Role---
You are a Knowledge Graph Specialist extracting entities and relationships specifically from IT-media posts, conference updates, video announcements, technical episodes, and news about technologies.

Your goal: extract clean structured entities and relationships from texts related to Flutter, Dart, conferences, tools, frameworks, AI-features, guests, episodes, releases, showcases and developer tooling.

---Instructions---

1. Entity Extraction
   Extract only **meaningful, real entities**:
   - Events (FlutterCon Europe 25)
   - People (speakers, guests)
   - Organizations (Google, channels)
   - Technologies (Flutter, Figma MCP)
   - Media (videos, episodes)
   - Topics (theming, typography)
   - Tools (plugins, analyzers)
   - Frameworks (Flutter, AngularDart)

Format:
entity{tuple_delimiter}entity_name{tuple_delimiter}entity_type{tuple_delimiter}entity_description

2. Relationship Extraction
Format:
relation{tuple_delimiter}source{tuple_delimiter}target{tuple_delimiter}keywords{tuple_delimiter}description

Allowed relations:
- participates_in
- hosted_by
- guest_in
- demonstrates
- uses
- explains
- covers
- related_to
- publishes
- mentions
- part_of

3. Rules:
- Output only in {language}
- Use EXACT entity names
- Output entities first, then relationships
- End with {completion_delimiter}

---Examples---

entity{tuple_delimiter}FlutterCon Europe 25{tuple_delimiter}event{tuple_delimiter}Major European Flutter conference in 2025.
entity{tuple_delimiter}Observable Flutter{tuple_delimiter}media{tuple_delimiter}Flutter weekly show.
entity{tuple_delimiter}Muhammad Hamza{tuple_delimiter}person{tuple_delimiter}Guest of the episode.
entity{tuple_delimiter}Figma MCP{tuple_delimiter}technology{tuple_delimiter}AI-powered design-system generator.

relation{tuple_delimiter}Observable Flutter{tuple_delimiter}Muhammad Hamza{tuple_delimiter}guest_in{tuple_delimiter}Guest of the episode.
relation{tuple_delimiter}Muhammad Hamza{tuple_delimiter}Figma MCP{tuple_delimiter}demonstrates{tuple_delimiter}Shows how Figma MCP works.

{completion_delimiter}

---Real Data---
Entity_types: [{entity_types}]
Text:
```

{input_text}

```
""",

    "entity_continue_extraction": """
---Role---
You continue extraction only for missing or incorrectly formatted items.

---Instructions---

1. DO NOT re-output previously correct items.
2. Output only:
   - Missing entities
   - Missing relationships
   - Incorrect/truncated items

3. Format MUST be identical:
   entity{tuple_delimiter}...(4 fields)
   relation{tuple_delimiter}...(5 fields)

4. End with {completion_delimiter}.
5. Output only data, no comments.

---Real Data---
Entity_types: [{entity_types}]
Text:
```

{input_text}

```
"""
}


# ---------------------------
# 2) LightRAG initialization
# ---------------------------

rag = LightRAG(
    working_dir="rag_data",
    addon_params=addon_params,
)


# ---------------------------
# 3) Sample texts (как твои)
# ---------------------------

texts = [
    "FlutterCon Europe 25 начался 🚀🚀🚀\nКстати организаторы обещали в этот раз появление докладов у них на ютуб канале прямо в день выступления.",
    
    "Итак, розыгрыш \"несуществующих билетов\" для сбора имейлов завершен. Результаты в комментариях к этому посту.",

    "Очередной выпуск Observable Flutter посвящён работе с AI... "
    "В качестве гостя — Muhammad Hamza, который демонстрирует Figma MCP...",
]


# ---------------------------
# 4) Async pipeline
# ---------------------------

async def main():
    print("\n### Добавление текстов в базу...")
    for t in texts:
        await rag.aadd(t)

    print("\n### Запуск асинхронного пайплайна...")
    await rag.apipeline()

    print("\n### Тестируем запрос...")
    response = await rag.aquery("Что такое Figma MCP?", QueryParam(mode="global"))
    print("\nОтвет:")
    print(response)

asyncio.run(main())
```

---

# ⚡ Что делать дальше?

Хочешь, я:

✅ добавлю автоматическое определение типа сущности
✅ добавлю улучшенные few-shot примеры
✅ добавлю post-processing для русского → английского нормализатора
✅ сделаю extractor специально под твой формат постов?


In [ ]:
addon_params = {
    "entity_extraction": """
---Goal---
Извлеки из текста все важные **сущности** и **отношения** между ними.

---Instructions---
1. Определи сущности в тексте. Возможные типы сущностей: event (событие), person (человек), technology (технология), organization (организатор), topic (тема), date (дата/время).  
2. Для каждой сущности выведи:
   - имя сущности (название)  
   - тип сущности  
   - короткое описание (в контексте данного текста)  
3. Определи отношения между сущностями, если они явно выражены. Например:
   - кто организует событие  
   - какая технология используется в контексте события  
   - о чём тема доклада  
4. Формат вывода:
   - `entity<|#|>имя<|#|>тип<|#|>описание`  
   - `relation<|#|>сущность_источник<|#|>сущность_цель<|#|>ключевые_слова<|#|>описание_отношения`  
5. Используй разделитель `<|#|>` и после всех записей добавь `<|COMPLETE|>`.

---Examples---
Пример 1:
```

entity<|#|>FlutterCon Europe 25<|#|>event<|#|>конференция Flutter, проходящая в Европе в 2025 году
entity<|#|>Muhammad Hamza<|#|>person<|#|>спикер, разработчик, работает с Figma MCP
entity<|#|>Figma MCP<|#|>technology<|#|>инструмент для темизации и генерации дизайн-систем с помощью нейросетей
relation<|#|>Muhammad Hamza<|#|>FlutterCon Europe 25<|#|>speaks_at<|#|>Muhammad выступает на FlutterCon Europe 25
relation<|#|>Figma MCP<|#|>Observable Flutter<|#|>used_in<|#|>Figma MCP применяется в Observable Flutter для создания дизайн-системы
<|COMPLETE|>

```

Пример 2:
```

entity<|#|>Observable Flutter Episode<|#|>event<|#|>выпуск шоу / видео о Flutter и AI
entity<|#|>AI<|#|>topic<|#|>искусственный интеллект
entity<|#|>дизайн‑система<|#|>topic<|#|>система дизайна приложения
relation<|#|>Observable Flutter Episode<|#|>covers<|#|>AI<|#|>выпуск посвящён теме AI
relation<|#|>Observable Flutter Episode<|#|>discusses<|#|>дизайн‑система<|#|>в выпуске обсуждается дизайн-система приложения
<|COMPLETE|>

```

""",
    "DEFAULT_TUPLE_DELIMITER": "<|#|>",
    "DEFAULT_COMPLETION_DELIMITER": "<|COMPLETE|>",
    # Можно добавить few-shot примеры, если нужно
    "entity_extraction_examples": [
        "entity<|#|>FlutterCon Europe 25<|#|>event<|#|>конференция Flutter, проходящая в Европе в 2025 году",
        "relation<|#|>Muhammad Hamza<|#|>FlutterCon Europe 25<|#|>speaks_at<|#|>Muhammad выступает на конференции"
    ]
}

Отличная задача. Вот пример шаблона (prompt) для LightRAG, адаптированного под **твой формат текстовок**, которые ты привёл (анонсы, новости про Flutter, AI и т.п.). Шаблон учитывает сущности типа “мероприятие”, “человек”, “продукт / технология”, “дата / время”, “тема” и отношения между ними (“организован”, “презентует”, “касается” и т.п.).

---

## 📚 Пример шаблона `addon_params` для LightRAG

```python
addon_params = {
    "entity_extraction": """
---Goal---
Извлеки из текста все важные **сущности** и **отношения** между ними.

---Instructions---
1. Определи сущности в тексте. Возможные типы сущностей: event (событие), person (человек), technology (технология), organization (организатор), topic (тема), date (дата/время).  
2. Для каждой сущности выведи:
   - имя сущности (название)  
   - тип сущности  
   - короткое описание (в контексте данного текста)  
3. Определи отношения между сущностями, если они явно выражены. Например:
   - кто организует событие  
   - какая технология используется в контексте события  
   - о чём тема доклада  
4. Формат вывода:
   - `entity<|#|>имя<|#|>тип<|#|>описание`  
   - `relation<|#|>сущность_источник<|#|>сущность_цель<|#|>ключевые_слова<|#|>описание_отношения`  
5. Используй разделитель `<|#|>` и после всех записей добавь `<|COMPLETE|>`.

---Examples---
Пример 1:
```

entity<|#|>FlutterCon Europe 25<|#|>event<|#|>конференция Flutter, проходящая в Европе в 2025 году
entity<|#|>Muhammad Hamza<|#|>person<|#|>спикер, разработчик, работает с Figma MCP
entity<|#|>Figma MCP<|#|>technology<|#|>инструмент для темизации и генерации дизайн-систем с помощью нейросетей
relation<|#|>Muhammad Hamza<|#|>FlutterCon Europe 25<|#|>speaks_at<|#|>Muhammad выступает на FlutterCon Europe 25
relation<|#|>Figma MCP<|#|>Observable Flutter<|#|>used_in<|#|>Figma MCP применяется в Observable Flutter для создания дизайн-системы
<|COMPLETE|>

```

Пример 2:
```

entity<|#|>Observable Flutter Episode<|#|>event<|#|>выпуск шоу / видео о Flutter и AI
entity<|#|>AI<|#|>topic<|#|>искусственный интеллект
entity<|#|>дизайн‑система<|#|>topic<|#|>система дизайна приложения
relation<|#|>Observable Flutter Episode<|#|>covers<|#|>AI<|#|>выпуск посвящён теме AI
relation<|#|>Observable Flutter Episode<|#|>discusses<|#|>дизайн‑система<|#|>в выпуске обсуждается дизайн-система приложения
<|COMPLETE|>

```

""",
    "DEFAULT_TUPLE_DELIMITER": "<|#|>",
    "DEFAULT_COMPLETION_DELIMITER": "<|COMPLETE|>",
    # Можно добавить few-shot примеры, если нужно
    "entity_extraction_examples": [
        "entity<|#|>FlutterCon Europe 25<|#|>event<|#|>конференция Flutter, проходящая в Европе в 2025 году",
        "relation<|#|>Muhammad Hamza<|#|>FlutterCon Europe 25<|#|>speaks_at<|#|>Muhammad выступает на конференции"
    ]
}
```

---

## ✅ Как использовать этот шаблон

Когда создаёшь `LightRAG`, передай `addon_params`:

```python
from lightrag import LightRAG

rag = LightRAG(
    working_dir="./data/lightrag",
    llm_model_func=твоя_llm_функция,
    addon_params=addon_params
)

await rag.initialize_storages()
await rag.ainsert(твой_текст)  # вставляешь текстыок
```

---

Если хочешь — я могу **сгенерировать 5–6 готовых prompt‑примеров** (few-shot) конкретно под твои тексты (анонсы FlutterCon, Observable Flutter и т.п.), чтобы LLM точно “понимал”, какие сущности и отношения ему нужно выделять. Сделать так?


In [ ]:
!pip install fast_graphrag "pydantic>=2.0.0"

In [ ]:
###############################################
# 6. Graph‑Based RAG Integration
###############################################

# 6.1 fast‑graphrag
# Install: pip install fast‑graphrag  or clone https://github.com/circlemind-ai/fast-graphrag  
from fast_graphrag import GraphRAG

grag = GraphRAG(
    working_dir="./data/fast_graphrag_storage",
    # возможны аргументы: domain, example_queries и т.д.
)

# insert documents (chunks)
for _, r in chunks_df.iterrows():
    grag.insert(r['text'])

# query
query = "Когда проходит конференция Flutter в Санкт‑Петербурге?"
resp = grag.query(query)
print("Fast‑GraphRAG Response:", resp.response)
print("Sources:", resp.sources)  

In [ ]:
!pip uninstall lightrag -y
!pip install git+https://github.com/HKUDS/LightRAG.git

In [ ]:
# Импорт для последней версии
from lightrag import LightRAG, QueryParam


In [ ]:
from lightrag.client import LightRAGClient
from lightrag.types import QueryParam


In [ ]:
from sentence_transformers import SentenceTransformer

# Embedding функция
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
def embedding_func(texts):
    # возвращает list[float] для каждого текста
    return embed_model.encode(texts, convert_to_numpy=True)

# LLM функция (stub / тестовая)
def llm_model_func(prompt: str):
    # можно интегрировать OpenAI или просто вернуть тестовый текст
    return "Ответ на ваш вопрос: …"


In [ ]:
# 6.2 LightRAG
# Install: pip install lightrag‑hku[api] or clone https://github.com/HKUDS/LightRAG  
import asyncio
from lightrag import LightRAG, QueryParam

async def run_light():
    rag = LightRAG(
        working_dir="./data/lightrag_storage",
        llm_model_func=llm_model_func,
        embedding_func=embedding_func
    )
    await rag.initialize_storages()
    await rag.insert_docs(chunks_df['text'].tolist())
    q = "Где можно купить билеты на конференцию?"
    result = await rag.query(q, param=QueryParam(mode="mix"))
    print("LightRAG Response:", result.response)
    print("LightRAG Chunks:", result.chunks)

asyncio.run(run_light())


In [ ]:
###############################################
# 6. Graph‑Based RAG Integration
###############################################

# 6.1 fast‑graphrag
# Install: pip install fast‑graphrag  or clone https://github.com/circlemind-ai/fast-graphrag  
from fast_graphrag import GraphRAG

grag = GraphRAG(
    working_dir="./fast_graphrag_storage",
    # возможны аргументы: domain, example_queries и т.д.
)

# insert documents (chunks)
for _, r in chunks_df.iterrows():
    grag.insert(r['text'])

# query
query = "Когда проходит конференция Flutter в Санкт‑Петербурге?"
resp = grag.query(query)
print("Fast‑GraphRAG Response:", resp.response)
print("Sources:", resp.sources)  

# 6.2 LightRAG
# Install: pip install lightrag‑hku[api] or clone https://github.com/HKUDS/LightRAG  
import asyncio
from lightrag import LightRAG, QueryParam

async def run_light():
    rag = LightRAG(
        working_dir="./lightrag_storage",
        llm_model_func=<ваш LLM функция>,
        embedding_func=<ваш embedding функция>
    )
    await rag.initialize_storages()
    await rag.insert_docs(chunks_df['text'].tolist())
    q = "Где можно купить билеты на конференцию?"
    result = await rag.query(q, param=QueryParam(mode="mix"))
    print("LightRAG Response:", result.response)
    print("LightRAG Chunks:", result.chunks)

asyncio.run(run_light())

# 6.3 MiniRAG
# Install: pip install minirag‑hku or clone https://github.com/HKUDS/MiniRAG  
from minirag import MiniRAG

mr = MiniRAG(
    data_path="path/to/chunks.json",   # подготовьте JSON с вашими чанками
    graph_type="heterogeneous",
    max_neighbors=5
)
mr.index()
q = "Расскажите о розыгрыше билетов онлайн и офлайн"
ans = mr.query(q)
print("MiniRAG Answer:", ans)


Отлично — сделаю развёрнутый план и готовый набор экспериментов, который вы сможете выполнить на ваших данных (пример фрейма данных приложили). Я опишу:

1. какие реализации/репозитории стоит сравнить (и почему);
2. какие метрики считать и почему (как именно считать — формулы/код);
3. готовые скрипты/псевдокод для: подготовки данных → построения графа → индексации → ретривала → генерации;
4. набор промптов и гиперпараметров для тестирования;
5. план экспериментов (матрица запусков) и рекомендации по анализу результатов.

Сразу важное: в экосистеме появились несколько готовых Graph-RAG/LightRAG/MiniRAG реализаций — рекомендую тестировать минимум 3 варианта (Fast/Light/Mini) плюс классический векторный RAG (baseline). Источники и реализации, с которыми я ориентирую рекомендации: Fast-GraphRAG (fast-graphrag / fastRAG), LightRAG, MiniRAG и общий обзор GraphRAG-ресурсов. ([GitHub][1])

---

# 1) Что сравниваем (кандидаты)

Коротко — какие системы запускать и почему:

* **Fast-GraphRAG / fast-graphrag** — ориентирован на скорость и масштаб, быстрые PPR/graph-based retrieval оптимизации. Хорош как промышленный baseline с графом. ([GitHub][1])
* **LightRAG** — простая и хорошо документированная система с UI/API, удобна для быстрого прототипирования и экспериментов с разными graph-фичами. ([GitHub][2])
* **MiniRAG** — «лёгкий» RAG для небольших моделей и edge-кейсов; полезен чтобы понять как граф влияет при малых моделях. ([GitHub][3])
* **Vector RAG (baseline)** — обычный pipeline: chunk → embedding (SentenceTransformers / Bert) → Qdrant → reranker (опционально) → LLM генерация. Нужен для сравнения «насколько граф даёт плюсы над векторным базовым подходом».

(Если хотите — можно также включить microsoft/graphrag, tiny-graphrag и другие репозитории как дополнительные варианты.) ([GitHub][4])

---

# 2) Выбор метрик — логика и расчёт

## 2.1. Ретривал (критично)

Для ретривала важны: **Recall@k**, **MRR** (Mean Reciprocal Rank), **Precision@k**, **nDCG@k**. Обоснование: эти метрики измеряют разное — recall измеряет покрытие релевантных документов (важно для RAG, чтобы нужная информация была найдена), MRR показывает ранжирование (важно для экономии токенов/качества ответа), nDCG учитывает ранжирование со взвешиванием по позиции (полезно когда есть graded relevance).

Формулы (код-реализация):

* Recall@k для запроса q:
  `recall@k_q = (# релевантных документов среди top-k) / (#релевантных документов в золотом)`
* MRR:
  `MRR = mean_q(1 / rank_first_relevant(q))` (если нет релевантного — 0)
* Precision@k:
  `precision@k_q = (#релевантных среди top-k) / k`
* nDCG@k:
  `DCG@k = sum_{i=1..k} (2^{rel_i} - 1) / log2(i+1)`;
  `nDCG@k = DCG@k / IDCG@k` (IDCG — идеальный DCG по золотому).

**Практическая заметка:** для коротких текстов (как у вас) релевантность можно задать бинарно: документ релевантен/не релевантен; если возможно — собрать аннотации (лучше) либо синтезировать «золотой» ответ через правила (например, если запрос содержит дату/тег/ключевое слово — пометить релевантными).

## 2.2. Генерация (фиделити / качество)

Метрики генерации делим на автоматические и интерактивную (human):

* **Automatic:** ROUGE-L, BLEU (для кратких ответов не лучший), BERTScore, BLEURT (если доступен), и QA-based factuality (см. ниже). BERTScore/BLEURT лучше подходят для семантического соответствия.
* **Factuality / Faithfulness (критично для RAG):** QA-based метрика: сгенерированный ответ разбивают на утверждения/факты → формулируют вопросы → проверяют ответы модели-распознавателя против источников (retrieved docs). Это даёт автоматическую оценку «факт/враньё». Также можно использовать entailment (NLI) между retrieved контекстом и ответом.
* **Human eval:** оценка по шкале (0–3) для: factuality, completeness, fluency, attribution (присутствует ли ссылка на источник).

## 2.3. Почему такой набор

* Ретривал-метрики измеряют «модель данных» — насколько хорошо система находит нужный контекст.
* Генерация — насколько LLM использует найденный контекст корректно и полно. Одна хорошая retrieval-метрика не гарантирует хорошую генерацию; поэтому нужны обе группы метрик и QA-based factuality как связующее звено.

---

# 3) Эксперимент — pipeline и готовый пример кода (псевдо/реалистичный Python)

Ниже — компактный, воспроизводимый pipeline, который вы можете сразу запустить/адаптировать. Я даю фрагменты: подготовка → векторный baseline → LightRAG/Fast/mini-интеграция (псевдо) → eval. Код — ориентировочно для локального запуска с `sentence-transformers`, `faiss`, `networkx`, `transformers`/OpenAI.

> Пример предполагает, что у вас есть `pandas` DataFrame `df` с колонками `id,date,text` (пример — ваш).

### 3.1. Подготовка данных (chunking, cleaning)

```python
import pandas as pd
from sentence_transformers import SentenceTransformer
import re
from sklearn.model_selection import train_test_split

# загрузка df
# df = pd.read_csv("your.csv", parse_dates=["date"])

def clean_text(t):
    t = re.sub(r'https?://\S+','', t)
    t = re.sub(r'\[(.*?)\]\(.*?\)', r'\1', t)  # markdown links
    return t.strip()

df['text_clean'] = df['text'].apply(clean_text)

# Простое chunking — по параграфам или N токенов
def chunk_text(s, max_chars=800):
    parts = []
    cur = ""
    for p in s.split("\n\n"):
        if len(cur) + len(p) < max_chars:
            cur += ("\n\n" + p) if cur else p
        else:
            if cur: parts.append(cur)
            cur = p
    if cur: parts.append(cur)
    return parts

rows = []
for _, r in df.iterrows():
    chunks = chunk_text(r['text_clean'])
    for i,c in enumerate(chunks):
        rows.append({"id": f"{r['id']}_{i}", "source_id": r['id'], "text": c, "date": r['date']})
chunks_df = pd.DataFrame(rows)
```

### 3.2. Векторный baseline: эмбеддинги + FAISS

```python
import faiss
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")  # пример
texts = chunks_df['text'].tolist()
embs = embed_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

d = embs.shape[1]
index = faiss.IndexHNSWFlat(d, 32)
index.hnsw.efConstruction = 200
faiss.normalize_L2(embs)
index.add(embs)

# функция поиска
def knn_search(query, k=5):
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D,I = index.search(q_emb, k)
    return I[0], D[0]
```

### 3.3. Простейший graph-enhanced retrieval (пример: entity graph + PPR)

Идея: строим граф где узлы — chunk'и и сущности; ребра — наличие сущности в chunk. Затем запускаем Personalized PageRank (PPR) с инициализацией на сущностях запроса.

```python
import networkx as nx
import spacy
nlp = spacy.load("xx_ent_wiki_sm")  # или ru модель

G = nx.Graph()
# добавляем узлы для каждого chunk
for idx, row in chunks_df.iterrows():
    G.add_node(row['id'], type='chunk', text=row['text'])

# извлечение сущностей (упрощённо)
for idx,row in chunks_df.iterrows():
    doc = nlp(row['text'])
    ents = set([ent.text for ent in doc.ents])
    for e in ents:
        if not G.has_node(e):
            G.add_node(e, type='entity')
        G.add_edge(row['id'], e)

# PPR search
def graph_rerank(query, top_k=5):
    q_doc = nlp(query)
    q_ents = [ent.text for ent in q_doc.ents]
    # если нет сущностей — fallback на векторный поиск
    if not q_ents:
        ids, d = knn_search(query, k=top_k*3)
        return [chunks_df.iloc[i]['text'] for i in ids[:top_k]]
    # personalization dict
    personalization = {n: 0.0 for n in G.nodes()}
    for e in q_ents:
        if e in G:
            personalization[e] = 1.0
    pr = nx.pagerank(G, alpha=0.85, personalization=personalization)
    # сортируем только chunk nodes
    chunk_scores = [(n, s) for n,s in pr.items() if G.nodes[n].get('type')=='chunk']
    chunk_scores.sort(key=lambda x: x[1], reverse=True)
    top_nodes = [n for n,_ in chunk_scores[:top_k]]
    return [G.nodes[n]['text'] for n in top_nodes]
```

> Этот простой пример демонстрирует один из подходов (entity graph + PPR). В реальных LightRAG/MiniRAG/fast-graphrag реализациях граф и retrieval сложнее (топология, weighting, multi-hop, reranking). ([GitHub][1])

### 3.4. Генерация — шаблон промпта (пример для OpenAI / local LLM)

```python
# Собираем retrieved контекст
retrieved = graph_rerank(query, top_k=5)  # либо knn_search
context = "\n\n---\n\n".join(retrieved)

prompt = f"""
You are an assistant. Use ONLY the information from the CONTEXT to answer the QUESTION.
CONTEXT:
{context}

QUESTION:
{query}

Answer concisely. If answer is not in CONTEXT say "I don't know — no info in the context."
Cite sources where possible by giving source chunk ids.
"""

# отправляем в LLM (pseudo)
# response = llm.generate(prompt)
```

---

# 4) Eval: как считать метрики на практике (код примеров)

### 4.1. Recall@k / Precision@k / MRR

Предполагается, что для каждого тест-запроса у вас есть список «золотых» релевантных `source_id` (например, id исходного документа/поста).

```python
def eval_retrieval(retriever_fn, queries, gold_ids, k=5):
    recalls, mrrs, precs = [], [], []
    for q, gold in zip(queries, gold_ids):
        retrieved_texts = retriever_fn(q, top_k=k)  # возвращаем список chunk texts или ids
        # если retriever возвращает тексты, маппим на source_id через chunks_df
        retrieved_ids = []
        for t in retrieved_texts:
            row = chunks_df[chunks_df['text']==t].iloc[0]
            retrieved_ids.append(row['source_id'])
        # recall
        hits = sum(1 for r in retrieved_ids if r in gold)
        recall = hits / max(1, len(gold))
        # precision
        prec = hits / k
        # MRR
        rank = next((i+1 for i, r in enumerate(retrieved_ids) if r in gold), None)
        rr = 1.0/rank if rank else 0.0
        recalls.append(recall); precs.append(prec); mrrs.append(rr)
    return {"recall@k": sum(recalls)/len(recalls),
            "precision@k": sum(precs)/len(precs),
            "MRR": sum(mrrs)/len(mrrs)}
```

### 4.2. Генерация: BERTScore / ROUGE / QA-based factuality

* ROUGE/BERTScore: используйте `rouge-score`, `bert-score` библиотеки.
* QA-based factuality (автомат.): для каждого сгенерированного утверждения автоматически формулируйте вопрос (или извлекайте факт как (subject, predicate, object)), задавайте его граф-retrieved контексту и сверяйте ответы. Если ответ совпадает — считаем факт подтверждённым. (Есть библиотеки/скрипты для этого, но их часто придётся имплементировать под домен.)

Пример вызова BERTScore:

```python
from bert_score import score
P, R, F1 = score(cands, refs, lang="ru" if ru_texts else "en")
```

---

# 5) Набор промптов и гиперпараметры для тестов

1. **Prompt A (conservative)** — "Use ONLY the CONTEXT. If not present, say 'no info'." — проверяет faithfulness (меньше халюцинаций).
2. **Prompt B (generative)** — "Answer using context but you may use background knowledge." — проверяет полноту/побочный генеративный контент.
3. **Prompt C (source-citing)** — просить LLM выдавать ответы с сноской на `source_id` (важно для attribution метрики).

Гиперпараметры:

* retrieval k: 1, 3, 5, 10
* reranker: включён/выключен (например, cross-encoder reranker)
* embedding model: `all-MiniLM-L6-v2`, `paraphrase-multilingual-mpnet-base-v2`, OpenAI embeddings (если доступны)
* LLM size: small local (Llama2-7B), mid (13B), API (gpt-4-style) — сравнить влияние качества генератора.

---

# 6) План экспериментов (матрица)

Рекомендуемая матрица (пример, можно расширить):

| ID  | Retriever | Embedding |  k |  Reranker |       LLM | Prompt   |
| --- | --------: | --------: | -: | --------: | --------: | -------- |
| B1  |    vector |    MiniLM |  5 |        no | Llama2-7B | Prompt A |
| B2  |    vector |    MiniLM |  5 | cross-enc | Llama2-7B | Prompt A |
| G1  |   FastRAG |    MiniLM |  5 |        no | Llama2-7B | Prompt A |
| L1  |  LightRAG |    MiniLM |  5 |        no | Llama2-7B | Prompt A |
| M1  |   MiniRAG |    MiniLM |  5 |        no | Llama2-7B | Prompt A |
| ... |       ... |       ... |    |           |           |          |

Для каждого запуска:

* измеряем retrieval метрики (Recall@k, MRR, nDCG@k)
* генерируем ответы на тест-сет вопросов → считаем ROUGE/BERTScore → QA-based factuality → human eval subset (~50 примеров)

---

# 7) Как собрать золотые данные / аннотации быстро

* Для начального теста используйте **weak supervision**: если в `text` есть ключевые слова из запроса — помечать как релевантный.
* Для более точной оценки собрать ручные аннотации хотя бы для 200 запросов (по 3–5 релевантных документов). Human eval можно разнести по нескольким метрикам (факт/полнота/плавность/атрибуция).

---

# 8) Анализ результатов — что смотреть

* Сравните recall@k и MRR между графовыми системами и baseline: если граф даёт заметный рост MRR при том же recall — он лучше ранжирует, что экономит токены LLM (меньше контекста).
* Сравните factuality (QA-based) и BERTScore для генерации: разрыв между высоким retrieval recall и низкой factuality указывает, что LLM не правильно использует контекст → проверьте промпты/отрывок контекста.
* Важная диаграмма: scatter plot (retrieval_recall vs factuality) — даёт понимание корреляции.

---

# 9) Практические рекомендации / чеклист перед запуском

* Подготовьте тест-набор запросов (50–500 запросов) с золотыми релевантными id.
* Запустите baseline (вектор) → зафиксируйте время индексации, размер индекса, среднее время запроса.
* Запустите Fast/Light/Mini варианты — соберите те же метрики + ресурсы/время.
* Для генерации: одинаковые prompts и LLM для всех экспериментов, иначе сравнение нечестно.
* Автоматизируйте подсчёт: CI скрипт, который будет сохранять все результаты в CSV/Parquet.

---

# 10) Примеры ожидаемых проблем и их устранение

* **Нет сущностей в коротких текcтах** → граф без сущностей бесполезен: добавить n-gram entity extraction или chunking по хэштегам/тегам.
* **Плохая релевантность — reranker нужен**: используйте cross-encoder reranker (много медленнее но точнее).
* **Халюцинации LLM**: строгий prompt «use ONLY context» + source-citing часто снижает их.

---

# 11) Что я могу сделать дальше (если хотите)

* Сгенерировать для вас **готовый Jupyter notebook** с вышеописанным pipeline под ваши данные (chunking → FAISS → basic graph → eval). (Если хотите — могу создать notebook и положить его в виде файла.)
* Запустить небольшой пробный эксперимент на ваших примерах (например, 20 запросов) и вернуть готовые графики/таблицы — для этого мне нужны: полный CSV или доступ к данным прямо в чате.

---

# 12) Источники (использованные при подготовке плана)

* Fast-GraphRAG (репозиторий / README). ([GitHub][1])
* LightRAG (репозиторий / сервер). ([GitHub][2])
* MiniRAG (репозиторий / arXiv). ([GitHub][3])
* Обзорные материалы / «Awesome-GraphRAG». ([GitHub][5])

---

Если нужно, сразу делаю одно из двух (выберите, не надо ждать — я выполню прямо сейчас в этом ответе):

A) Сгенерировать **готовый Jupyter notebook** (код + инструкции, который можно запустить локально) — вы получите файл .ipynb;
B) Подготовить **маленький прототип** (в виде runnable Python-скрипта здесь) и запустить его на ваших предоставленных данных (вам нужно вставить полный CSV прямо в чат или дать ключ/доступ — но я не могу запускать внешние сервисы).

Что предпочитаете? (если выбираете A — я сформирую notebook и дам инструкции; если B — вставьте полный CSV/ссылку на файл прямо в чат и я сгенерирую скрипт/ноутбук).

[1]: https://github.com/circlemind-ai/fast-graphrag?utm_source=chatgpt.com "circlemind-ai/fast-graphrag: RAG that intelligently adapts ..."
[2]: https://github.com/HKUDS/LightRAG?utm_source=chatgpt.com "[EMNLP2025] \"LightRAG: Simple and Fast Retrieval- ..."
[3]: https://github.com/HKUDS/MiniRAG?utm_source=chatgpt.com "MiniRAG: Making RAG Simpler with Small and Open- ..."
[4]: https://github.com/microsoft/graphrag?utm_source=chatgpt.com "microsoft/graphrag: A modular graph-based Retrieval ..."
[5]: https://github.com/DEEP-PolyU/Awesome-GraphRAG?utm_source=chatgpt.com "DEEP-PolyU/Awesome-GraphRAG"
